In [ ]:
# Install required package
!pip install pillow-heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 32.8 MB/s eta 0:00:00


In [ ]:
# Image format adjustment
import os
import glob
from PIL import Image
from pillow_heif import register_heif_opener

# Register HEIC opener with PIL so it can recognize the format
register_heif_opener()

folder_path = '/content/drive/MyDrive/sticker_project/'
image_files = glob.glob(os.path.join(folder_path, '*.jpeg')) + glob.glob(os.path.join(folder_path, '*.jpg'))

print("Starting conversion of Apple HEIC format to standard JPEG...")

fixed_count = 0
for file_path in image_files:
    file_name = os.path.basename(file_path)
    try:
        # Open the image (PIL can now read HEIC)
        with Image.open(file_path) as img:
            # Convert to standard RGB color mode
            img_rgb = img.convert('RGB')
            # Save as a true JPEG format, overwriting the original file
            img_rgb.save(file_path, format='JPEG')
            print(f"✅ Successfully converted and fixed: {file_name}")
            fixed_count += 1
    except Exception as e:
        print(f"❌ Conversion failed for: {file_name}, Error: {e}")

print(f"\nAwesome! Batch conversion completed! Successfully fixed {fixed_count} images.")

In [ ]:
# ==========================================
# SCRIPT 1: BASELINE ONLY
# ==========================================
import os
import json
import re
import torch
from diffusers import StableDiffusionPipeline
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    # Output path for baseline images (建议单独建一个baseline文件夹)
    output_folder = "/content/drive/MyDrive/Stickers_Results/baseline"

    # Only need the baseline JSON
    json_path_baseline = "/content/drive/MyDrive/batch_sticker_prompts_baseline.json"

    pretrained_model_name = "runwayml/stable-diffusion-v1-5"
    num_inference_steps = 50
    guidance_scale = 7.5

def extract_number_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return match.group(1)
    return None

def load_prompts_to_dict(json_path):
    if not os.path.exists(json_path):
        print(f"❌ JSON not found: {json_path}")
        return {}

    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    if isinstance(prompts_data, dict):
        prompts_list = list(prompts_data.values())
    else:
        prompts_list = prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']

    return prompt_dict

def main():
    config = Config()

    print(f"📥 Loading base model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.pretrained_model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to("cuda")
    pipe.enable_attention_slicing()
    print(f"✅ Base model loaded")

    os.makedirs(config.output_folder, exist_ok=True)

    print("📥 Loading baseline prompts...")
    prompts_baseline = load_prompts_to_dict(config.json_path_baseline)

    if not prompts_baseline:
        print("❌ Baseline prompt list is empty. Exiting.")
        return

    numbers = sorted(list(prompts_baseline.keys()))
    total = len(numbers)
    success_baseline = 0

    print(f"\n{'='*60}")
    print(f"🚀 Generating True Baseline Images (NO LoRA)")
    print(f"{'='*60}\n")

    for i, number in enumerate(numbers, start=1):
        prompt_baseline = prompts_baseline[number]

        # 统一命名为 animal_xxx_baseline.png
        output_file_baseline = f"animal_{number}_baseline.png"
        save_path_baseline = os.path.join(config.output_folder, output_file_baseline)

        print(f"[{i}/{total}] Baseline for animal_{number}.jpg")
        print(f"  📝 Prompt: {prompt_baseline}")

        try:
            image_baseline = pipe(
                prompt_baseline,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
            ).images[0]
            image_baseline.save(save_path_baseline)
            print(f"  ✅ Saved -> {output_file_baseline}\n")
            success_baseline += 1
        except Exception as e:
            print(f"  ❌ Failed: {e}\n")

    print(f"{'='*60}")
    print(f"🎉 Baseline Tasks Complete! Success: {success_baseline}/{total}")
    print(f"📁 Output Folder: {config.output_folder}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

✅ Base model loaded
📥 Loading baseline prompts...

🚀 Generating True Baseline Images (NO LoRA)

[1/37] Baseline for animal_1.jpg
  📝 Prompt: A sleek black cat sitting. Sticker style.


...

  ✅ Saved -> animal_7_baseline.png

[36/37] Baseline for animal_8.jpg
  📝 Prompt: A curious dog sitting and looking at the camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> animal_8_baseline.png

[37/37] Baseline for animal_9.jpg
  📝 Prompt: A fluffy black and white Border Collie dog sitting and looking at camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> animal_9_baseline.png

🎉 Baseline Tasks Complete! Success: 37/37
📁 Output Folder: /content/drive/MyDrive/Stickers_Results/baseline


In [ ]:
# ==========================================
# SCRIPT 2: FLAT LORA ONLY
# ==========================================
import os
import json
import re
import torch
from diffusers import StableDiffusionPipeline
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    lora_path = "/content/drive/MyDrive/lora_flat"

    # Output path for flat images
    output_folder = "/content/drive/MyDrive/Stickers_Results/flat"

    # Only need the main JSON
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    pretrained_model_name = "runwayml/stable-diffusion-v1-5"
    num_inference_steps = 50
    guidance_scale = 7.5

def load_lora_with_peft(pipe, lora_path):
    try:
        from peft import PeftModel
        print(f"\n🔄 Loading LoRA from: {lora_path}")
        pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)
        print(f"✅ LoRA loaded successfully")
        return True
    except Exception as e:
        print(f"❌ Failed to load LoRA: {e}")
        return False

def extract_number_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return match.group(1)
    return None

def load_prompts_to_dict(json_path):
    if not os.path.exists(json_path):
        print(f"❌ JSON not found: {json_path}")
        return {}

    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    if isinstance(prompts_data, dict):
        prompts_list = list(prompts_data.values())
    else:
        prompts_list = prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']

    return prompt_dict

def main():
    config = Config()

    print(f"📥 Loading base model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.pretrained_model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to("cuda")
    pipe.enable_attention_slicing()
    print(f"✅ Base model loaded")

    os.makedirs(config.output_folder, exist_ok=True)

    print("📥 Loading main prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    if not prompts_main:
        print("❌ Main prompt list is empty. Exiting.")
        return

    numbers = sorted(list(prompts_main.keys()))
    total = len(numbers)
    success_lora = 0

    print(f"\n{'='*60}")
    print(f"🚀 Generating Flat LoRA Images")
    print(f"{'='*60}")

    # Load LoRA immediately
    if not load_lora_with_peft(pipe, config.lora_path):
        print("❌ Cannot proceed without LoRA. Exiting.")
        return

    print("\n")
    for i, number in enumerate(numbers, start=1):
        prompt_main = prompts_main[number]

        output_file_main = f"flat_animal_{number}.png"
        save_path_main = os.path.join(config.output_folder, output_file_main)

        print(f"[{i}/{total}] LoRA for animal_{number}.jpg")
        print(f"  📝 Prompt: {prompt_main}")

        try:
            image_main = pipe(
                prompt_main,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
            ).images[0]
            image_main.save(save_path_main)
            print(f"  ✅ Saved -> {output_file_main}\n")
            success_lora += 1
        except Exception as e:
            print(f"  ❌ Failed: {e}\n")

    print(f"{'='*60}")
    print(f"🎉 LoRA Tasks Complete! Success: {success_lora}/{total}")
    print(f"📁 Output Folder: {config.output_folder}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

✅ Base model loaded
📥 Loading main prompts...

🚀 Generating Flat LoRA Images

🔄 Loading LoRA from: /content/drive/MyDrive/lora_flat
✅ LoRA loaded successfully


[1/37] LoRA for animal_1.jpg
  📝 Prompt: A sleek black sks cat sitting. Sticker style.


...

  ✅ Saved -> flat_animal_7.png

[36/37] LoRA for animal_8.jpg
  📝 Prompt: A curious sks dog sitting and looking at the camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> flat_animal_8.png

[37/37] LoRA for animal_9.jpg
  📝 Prompt: A fluffy black and white Border Collie sks dog sitting and looking at camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> flat_animal_9.png

🎉 LoRA Tasks Complete! Success: 37/37
📁 Output Folder: /content/drive/MyDrive/Stickers_Results/flat


In [ ]:
# ==========================================
# SCRIPT 2: FLAT LORA ONLY (Added Fixed Seed)
# ==========================================
import os
import json
import re
import torch
from diffusers import StableDiffusionPipeline
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    lora_path = "/content/drive/MyDrive/lora_flat"

    # Output path for flat images
    output_folder = "/content/drive/MyDrive/Stickers_Results/flat"

    # Only need the main JSON
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    pretrained_model_name = "runwayml/stable-diffusion-v1-5"
    num_inference_steps = 50
    guidance_scale = 7.5

    # 👇 新增：设置一个固定的随机种子
    seed = 42

def load_lora_with_peft(pipe, lora_path):
    try:
        from peft import PeftModel
        print(f"\n🔄 Loading LoRA from: {lora_path}")
        pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)
        print(f"✅ LoRA loaded successfully")
        return True
    except Exception as e:
        print(f"❌ Failed to load LoRA: {e}")
        return False

def extract_number_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return match.group(1)
    return None

def load_prompts_to_dict(json_path):
    if not os.path.exists(json_path):
        print(f"❌ JSON not found: {json_path}")
        return {}

    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    if isinstance(prompts_data, dict):
        prompts_list = list(prompts_data.values())
    else:
        prompts_list = prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']

    return prompt_dict

def main():
    config = Config()

    print(f"📥 Loading base model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.pretrained_model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to("cuda")
    pipe.enable_attention_slicing()
    print(f"✅ Base model loaded")

    os.makedirs(config.output_folder, exist_ok=True)

    print("📥 Loading main prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    if not prompts_main:
        print("❌ Main prompt list is empty. Exiting.")
        return

    numbers = sorted(list(prompts_main.keys()))
    total = len(numbers)
    success_lora = 0

    print(f"\n{'='*60}")
    print(f"🚀 Generating Flat LoRA Images (Fixed Seed: {config.seed})")
    print(f"{'='*60}")

    # Load LoRA immediately
    if not load_lora_with_peft(pipe, config.lora_path):
        print("❌ Cannot proceed without LoRA. Exiting.")
        return

    print("\n")
    for i, number in enumerate(numbers, start=1):
        prompt_main = prompts_main[number]

        output_file_main = f"flat_animal_{number}.png"
        save_path_main = os.path.join(config.output_folder, output_file_main)

        print(f"[{i}/{total}] LoRA for animal_{number}.jpg")
        print(f"  📝 Prompt: {prompt_main}")

        # 👇 新增：每次生成前，使用固定的种子创建生成器
        generator = torch.Generator(device="cuda").manual_seed(config.seed)

        try:
            image_main = pipe(
                prompt_main,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
                generator=generator # 👇 新增：把生成器传给 pipeline
            ).images[0]
            image_main.save(save_path_main)
            print(f"  ✅ Saved -> {output_file_main}\n")
            success_lora += 1
        except Exception as e:
            print(f"  ❌ Failed: {e}\n")

    print(f"{'='*60}")
    print(f"🎉 LoRA Tasks Complete! Success: {success_lora}/{total}")
    print(f"📁 Output Folder: {config.output_folder}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Mounted at /content/drive
📥 Loading base model...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

...

  ✅ Saved -> flat_animal_7.png

[36/37] LoRA for animal_8.jpg
  📝 Prompt: A curious sks dog sitting and looking at the camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> flat_animal_8.png

[37/37] LoRA for animal_9.jpg
  📝 Prompt: A fluffy black and white Border Collie sks dog sitting and looking at camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> flat_animal_9.png

🎉 LoRA Tasks Complete! Success: 37/37
📁 Output Folder: /content/drive/MyDrive/Stickers_Results/flat


In [ ]:
# ==========================================
# SCRIPT 2: FLAT LORA ONLY (Selective Overwrite Version)
# ==========================================
import os
import json
import re
import torch
from diffusers import StableDiffusionPipeline
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    lora_path = "/content/drive/MyDrive/lora_flat"

    # Output path for flat images
    output_folder = "/content/drive/MyDrive/Stickers_Results/flat"

    # Only need the main JSON
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    pretrained_model_name = "runwayml/stable-diffusion-v1-5"
    num_inference_steps = 50
    guidance_scale = 7.5

    # 🌟 CORE FEATURE: Enter the image numbers you want to regenerate here (as strings).
    # For example, to regenerate animal_2, animal_5, and animal_12, use ["2", "5", "12"].
    # If you leave it empty [], it will default to generating ALL images in the JSON.
    target_numbers = ["15"]

def load_lora_with_peft(pipe, lora_path):
    try:
        from peft import PeftModel
        print(f"\n🔄 Loading LoRA from: {lora_path}")
        pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)
        print(f"✅ LoRA loaded successfully")
        return True
    except Exception as e:
        print(f"❌ Failed to load LoRA: {e}")
        return False

def extract_number_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return match.group(1)
    return None

def load_prompts_to_dict(json_path):
    if not os.path.exists(json_path):
        print(f"❌ JSON not found: {json_path}")
        return {}

    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    if isinstance(prompts_data, dict):
        prompts_list = list(prompts_data.values())
    else:
        prompts_list = prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']

    return prompt_dict

def main():
    config = Config()

    print(f"📥 Loading base model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.pretrained_model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to("cuda")
    pipe.enable_attention_slicing()
    print(f"✅ Base model loaded")

    os.makedirs(config.output_folder, exist_ok=True)

    print("📥 Loading main prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    if not prompts_main:
        print("❌ Main prompt list is empty. Exiting.")
        return

    # 🌟 CORE LOGIC: Filter the numbers to be regenerated
    if config.target_numbers and len(config.target_numbers) > 0:
        # Keep only the numbers that exist in the JSON AND are specified in target_numbers
        numbers = [n for n in config.target_numbers if n in prompts_main]
        print(f"\n🎯 Selective generation mode enabled! Only the following numbers will be regenerated: {numbers}")
    else:
        numbers = sorted(list(prompts_main.keys()))
        print(f"\n🎯 No specific numbers provided. Generating all {len(numbers)} images.")

    if not numbers:
        print("❌ No valid numbers found to generate. Please check the target_numbers setting or the JSON file.")
        return

    total = len(numbers)
    success_lora = 0

    print(f"\n{'='*60}")
    print(f"🚀 Generating Flat LoRA Images (Selective Overwrite Mode)")
    print(f"{'='*60}")

    # Load LoRA immediately
    if not load_lora_with_peft(pipe, config.lora_path):
        print("❌ Cannot proceed without LoRA. Exiting.")
        return

    print("\n")
    for i, number in enumerate(numbers, start=1):
        prompt_main = prompts_main[number]

        output_file_main = f"flat_animal_{number}.png"
        save_path_main = os.path.join(config.output_folder, output_file_main)

        print(f"[{i}/{total}] LoRA for animal_{number}.jpg")
        print(f"  📝 Prompt: {prompt_main}")

        try:
            # Generate the image
            image_main = pipe(
                prompt_main,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
            ).images[0]

            # Check if file exists to notify the user about the overwrite
            if os.path.exists(save_path_main):
                print(f"  ⚠️ Old file found, overwriting -> {output_file_main}")
            else:
                print(f"  ✨ Creating new file -> {output_file_main}")

            # Save (this automatically overwrites if the file exists)
            image_main.save(save_path_main)
            print(f"  ✅ Saved successfully!\n")
            success_lora += 1
        except Exception as e:
            print(f"  ❌ Failed: {e}\n")

    print(f"{'='*60}")
    print(f"🎉 LoRA Tasks Complete! Success: {success_lora}/{total}")
    print(f"📁 Output Folder: {config.output_folder}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

✅ Base model loaded
📥 Loading main prompts...

🎯 Selective generation mode enabled! Only the following numbers will be regenerated: ['15']

🚀 Generating Flat LoRA Images (Selective Overwrite Mode)

🔄 Loading LoRA from: /content/drive/MyDrive/lora_flat
✅ LoRA loaded successfully


[1/1] LoRA for animal_15.jpg
  📝 Prompt: A curious gray sks cat looking at the camera with its eyes wide open, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ⚠️ Old file found, overwriting -> flat_animal_15.png
  ✅ Saved successfully!

🎉 LoRA Tasks Complete! Success: 1/1
📁 Output Folder: /content/drive/MyDrive/Stickers_Results/flat


In [ ]:
# cartoon
import os
import json
import re
import torch
from diffusers import StableDiffusionPipeline
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    drive_path = "/content/drive/MyDrive/sticker_project"
    lora_path = "/content/drive/MyDrive/lora_cartoon"

    # Output path for generated images
    output_folder = "/content/drive/MyDrive/Stickers_Results/cartoon"

    # Path for the main JSON file (we don't need the baseline JSON anymore)
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    pretrained_model_name = "runwayml/stable-diffusion-v1-5"

    num_inference_steps = 50
    guidance_scale = 7.5

def load_lora_with_peft(pipe, lora_path):
    """Load LoRA using diffusers native method"""
    try:
        print(f"\n🔄 Loading LoRA from: {lora_path}")
        pipe.load_lora_weights(lora_path)
        print(f"✅ LoRA loaded successfully")
        return True
    except Exception as e:
        print(f"❌ Failed to load LoRA: {e}")
        return False

def extract_number_from_filename(filename):
    """Extract the numeric ID from the filename"""
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return match.group(1)
    return None

def load_prompts_to_dict(json_path):
    """Read JSON and convert to a dictionary with the extracted number as the key"""
    if not os.path.exists(json_path):
        print(f"❌ JSON not found: {json_path}")
        return {}

    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    if isinstance(prompts_data, dict):
        prompts_list = list(prompts_data.values())
    else:
        prompts_list = prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']

    return prompt_dict

def main():
    config = Config()

    # 1. Load base model
    print(f"📥 Loading base model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.pretrained_model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to("cuda")

    pipe.enable_attention_slicing()
    print(f"✅ Base model loaded")

    # Create output directory
    os.makedirs(config.output_folder, exist_ok=True)

    # Load prompts
    print("📥 Loading prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    if not prompts_main:
        print("❌ Main prompt list is empty or failed to load. Exiting.")
        return

    numbers = sorted(list(prompts_main.keys()))
    total = len(numbers)
    success_lora = 0

    # ==========================================
    # Generate LoRA Images (WITH LoRA)
    # ==========================================
    print(f"\n{'='*60}")
    print(f"🚀 Generating Cartoon LoRA Images")
    print(f"{'='*60}")

    # Load LoRA immediately
    if not load_lora_with_peft(pipe, config.lora_path):
        print("❌ Cannot proceed without LoRA. Exiting.")
        return

    print("\n")
    for i, number in enumerate(numbers, start=1):
        prompt_main = prompts_main[number]

        output_file_main = f"cartoon_animal_{number}.png"
        save_path_main = os.path.join(config.output_folder, output_file_main)

        print(f"[{i}/{total}] LoRA for animal_{number}.jpg")
        print(f"  📝 Prompt: {prompt_main}")

        try:
            image_main = pipe(
                prompt_main,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
            ).images[0]
            image_main.save(save_path_main)
            print(f"  ✅ Saved -> {output_file_main}\n")
            success_lora += 1
        except Exception as e:
            print(f"  ❌ Failed: {e}\n")

    # Statistics
    print(f"{'='*60}")
    print(f"🎉 All Tasks Complete!")
    print(f"✅ Cartoon LoRA Success: {success_lora}/{total}")
    print(f"📁 Output Folder: {config.output_folder}")
    print(f"{'='*60}")

# Run the script
if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

✅ Base model loaded
📥 Loading prompts...

🚀 Generating Cartoon LoRA Images

🔄 Loading LoRA from: /content/drive/MyDrive/lora_cartoon


...

  ✅ Saved -> cartoon_animal_7.png

[36/37] LoRA for animal_8.jpg
  📝 Prompt: A curious sks dog sitting and looking at the camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> cartoon_animal_8.png

[37/37] LoRA for animal_9.jpg
  📝 Prompt: A fluffy black and white Border Collie sks dog sitting and looking at camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> cartoon_animal_9.png

🎉 All Tasks Complete!
✅ Cartoon LoRA Success: 37/37
📁 Output Folder: /content/drive/MyDrive/Stickers_Results/cartoon


In [ ]:
# ==========================================
# SCRIPT: CARTOON LORA ONLY (Selective Overwrite Version)
# ==========================================
import os
import json
import re
import torch
from diffusers import StableDiffusionPipeline
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    drive_path = "/content/drive/MyDrive/sticker_project"
    lora_path = "/content/drive/MyDrive/lora_cartoon"

    # Output path for generated images
    output_folder = "/content/drive/MyDrive/Stickers_Results/cartoon"

    # Path for the main JSON file
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    pretrained_model_name = "runwayml/stable-diffusion-v1-5"

    num_inference_steps = 50
    guidance_scale = 7.5

    # 🌟 CORE FEATURE: Enter the image numbers you want to regenerate here (as strings).
    # For example, to regenerate animal_2, animal_5, and animal_12, use ["2", "5", "12"].
    # If you leave it empty [], it will default to generating ALL images in the JSON.
    target_numbers = ["33"]

def load_lora(pipe, lora_path):
    """Load LoRA using diffusers native method (Fixed 'peft_type' error)"""
    try:
        print(f"\n🔄 Loading LoRA from: {lora_path}")
        pipe.load_lora_weights(lora_path)
        print(f"✅ LoRA loaded successfully")
        return True
    except Exception as e:
        print(f"❌ Failed to load LoRA: {e}")
        return False

def extract_number_from_filename(filename):
    """Extract the numeric ID from the filename"""
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return match.group(1)
    return None

def load_prompts_to_dict(json_path):
    """Read JSON and convert to a dictionary with the extracted number as the key"""
    if not os.path.exists(json_path):
        print(f"❌ JSON not found: {json_path}")
        return {}

    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    if isinstance(prompts_data, dict):
        prompts_list = list(prompts_data.values())
    else:
        prompts_list = prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']

    return prompt_dict

def main():
    config = Config()

    # 1. Load base model
    print(f"📥 Loading base model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.pretrained_model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to("cuda")

    pipe.enable_attention_slicing()
    print(f"✅ Base model loaded")

    # Create output directory
    os.makedirs(config.output_folder, exist_ok=True)

    # Load prompts
    print("📥 Loading prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    if not prompts_main:
        print("❌ Main prompt list is empty or failed to load. Exiting.")
        return

    # 🌟 CORE LOGIC: Filter the numbers to be regenerated
    if config.target_numbers and len(config.target_numbers) > 0:
        # Keep only the numbers that exist in the JSON AND are specified in target_numbers
        numbers = [n for n in config.target_numbers if n in prompts_main]
        print(f"\n🎯 Selective generation mode enabled! Only the following numbers will be regenerated: {numbers}")
    else:
        numbers = sorted(list(prompts_main.keys()))
        print(f"\n🎯 No specific numbers provided. Generating all {len(numbers)} images.")

    if not numbers:
        print("❌ No valid numbers found to generate. Please check the target_numbers setting or the JSON file.")
        return

    total = len(numbers)
    success_lora = 0

    # ==========================================
    # Generate LoRA Images (WITH LoRA)
    # ==========================================
    print(f"\n{'='*60}")
    print(f"🚀 Generating Cartoon LoRA Images (Selective Overwrite Mode)")
    print(f"{'='*60}")

    # Load LoRA immediately
    if not load_lora(pipe, config.lora_path):
        print("❌ Cannot proceed without LoRA. Exiting.")
        return

    print("\n")
    for i, number in enumerate(numbers, start=1):
        prompt_main = prompts_main[number]

        output_file_main = f"cartoon_animal_{number}.png"
        save_path_main = os.path.join(config.output_folder, output_file_main)

        print(f"[{i}/{total}] LoRA for animal_{number}.jpg")
        print(f"  📝 Prompt: {prompt_main}")

        try:
            # Generate the image
            image_main = pipe(
                prompt_main,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
            ).images[0]

            # Check if file exists to notify the user about the overwrite
            if os.path.exists(save_path_main):
                print(f"  ⚠️ Old file found, overwriting -> {output_file_main}")
            else:
                print(f"  ✨ Creating new file -> {output_file_main}")

            # Save (this automatically overwrites if the file exists)
            image_main.save(save_path_main)
            print(f"  ✅ Saved successfully!\n")
            success_lora += 1
        except Exception as e:
            print(f"  ❌ Failed: {e}\n")

    # Statistics
    print(f"{'='*60}")
    print(f"🎉 All Tasks Complete!")
    print(f"✅ Cartoon LoRA Success: {success_lora}/{total}")
    print(f"📁 Output Folder: {config.output_folder}")
    print(f"{'='*60}")

# Run the script
if __name__ == "__main__":
    main()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Mounted at /content/drive
📥 Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

✅ Base model loaded
📥 Loading prompts...

🎯 Selective generation mode enabled! Only the following numbers will be regenerated: ['33']

🚀 Generating Cartoon LoRA Images (Selective Overwrite Mode)

🔄 Loading LoRA from: /content/drive/MyDrive/lora_cartoon
❌ Failed to load LoRA: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/lora_cartoon'. Use `repo_type` argument if needed.
❌ Cannot proceed without LoRA. Exiting.


In [ ]:
# ==========================================
# SCRIPT: WATERCOLOR LORA ONLY (Selective Overwrite Version)
# ==========================================
import os
import json
import re
import torch
from diffusers import StableDiffusionPipeline
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    drive_path = "/content/drive/MyDrive/sticker_project"
    lora_path = "/content/drive/MyDrive/lora_watercolor"

    # Output path for generated images
    output_folder = "/content/drive/MyDrive/Stickers_Results/watercolor"

    # Path for the main JSON file
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    pretrained_model_name = "runwayml/stable-diffusion-v1-5"

    num_inference_steps = 50
    guidance_scale = 7.5

    # 🌟 CORE FEATURE: Enter the image numbers you want to regenerate here (as strings).
    # For example, to regenerate animal_2, animal_5, and animal_12, use ["2", "5", "12"].
    # If you leave it empty [], it will default to generating ALL images in the JSON.
    target_numbers = ["1","2","3","7"]

def load_lora(pipe, lora_path):
    """Load LoRA using diffusers native method"""
    try:
        print(f"\n🔄 Loading LoRA from: {lora_path}")
        pipe.load_lora_weights(lora_path)
        print(f"✅ LoRA loaded successfully")
        return True
    except Exception as e:
        print(f"❌ Failed to load LoRA: {e}")
        return False

def extract_number_from_filename(filename):
    """Extract the numeric ID from the filename"""
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return match.group(1)
    return None

def load_prompts_to_dict(json_path):
    """Read JSON and convert to a dictionary with the extracted number as the key"""
    if not os.path.exists(json_path):
        print(f"❌ JSON not found: {json_path}")
        return {}

    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    if isinstance(prompts_data, dict):
        prompts_list = list(prompts_data.values())
    else:
        prompts_list = prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']

    return prompt_dict

def main():
    config = Config()

    # 1. Load base model
    print(f"📥 Loading base model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.pretrained_model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to("cuda")

    pipe.enable_attention_slicing()
    print(f"✅ Base model loaded")

    # Create output directory
    os.makedirs(config.output_folder, exist_ok=True)

    # Load prompts
    print("📥 Loading prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    if not prompts_main:
        print("❌ Main prompt list is empty or failed to load. Exiting.")
        return

    # 🌟 CORE LOGIC: Filter the numbers to be regenerated
    if config.target_numbers and len(config.target_numbers) > 0:
        # Keep only the numbers that exist in the JSON AND are specified in target_numbers
        numbers = [n for n in config.target_numbers if n in prompts_main]
        print(f"\n🎯 Selective generation mode enabled! Only the following numbers will be regenerated: {numbers}")
    else:
        numbers = sorted(list(prompts_main.keys()))
        print(f"\n🎯 No specific numbers provided. Generating all {len(numbers)} images.")

    if not numbers:
        print("❌ No valid numbers found to generate. Please check the target_numbers setting or the JSON file.")
        return

    total = len(numbers)
    success_lora = 0

    # ==========================================
    # Generate LoRA Images (WITH LoRA)
    # ==========================================
    print(f"\n{'='*60}")
    print(f"🚀 Generating Watercolor LoRA Images")
    print(f"{'='*60}")

    # Load LoRA immediately
    if not load_lora(pipe, config.lora_path):
        print("❌ Cannot proceed without LoRA. Exiting.")
        return

    print("\n")
    for i, number in enumerate(numbers, start=1):
        prompt_main = prompts_main[number]

        # 修正了这里的文件名：cartoon_animal -> watercolor_animal
        output_file_main = f"watercolor_animal_{number}.png"
        save_path_main = os.path.join(config.output_folder, output_file_main)

        print(f"[{i}/{total}] LoRA for animal_{number}.jpg")
        print(f"  📝 Prompt: {prompt_main}")

        try:
            image_main = pipe(
                prompt_main,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
            ).images[0]

            # Check if file exists to notify the user about the overwrite
            if os.path.exists(save_path_main):
                print(f"  ⚠️ Old file found, overwriting -> {output_file_main}")
            else:
                print(f"  ✨ Creating new file -> {output_file_main}")

            image_main.save(save_path_main)
            print(f"  ✅ Saved -> {output_file_main}\n")
            success_lora += 1
        except Exception as e:
            print(f"  ❌ Failed: {e}\n")

    # Statistics
    print(f"{'='*60}")
    print(f"🎉 All Tasks Complete!")
    print(f"✅ Watercolor LoRA Success: {success_lora}/{total}")
    print(f"📁 Output Folder: {config.output_folder}")
    print(f"{'='*60}")

# Run the script
if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

✅ Base model loaded
📥 Loading prompts...

🎯 Selective generation mode enabled! Only the following numbers will be regenerated: ['1', '2', '3', '7']

🚀 Generating Watercolor LoRA Images

🔄 Loading LoRA from: /content/drive/MyDrive/lora_watercolor


...

  ⚠️ Old file found, overwriting -> watercolor_animal_2.png
  ✅ Saved -> watercolor_animal_2.png

[3/4] LoRA for animal_3.jpg
  📝 Prompt: Two sks black cats, one sitting and the other lying on its back, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ⚠️ Old file found, overwriting -> watercolor_animal_3.png
  ✅ Saved -> watercolor_animal_3.png

[4/4] LoRA for animal_7.jpg
  📝 Prompt: A fluffy white sks dog standing and looking at the camera, sticker style.


  0%|          | 0/50 [00:00<?, ?it/s]

  ⚠️ Old file found, overwriting -> watercolor_animal_7.png
  ✅ Saved -> watercolor_animal_7.png

🎉 All Tasks Complete!
✅ Watercolor LoRA Success: 4/4
📁 Output Folder: /content/drive/MyDrive/Stickers_Results/watercolor


In [4]:
# ==========================================
# SCRIPT: CREATE COMPARISON COLLAGES (5 Images per Row)
# ==========================================
import os
import math
from PIL import Image, ImageDraw, ImageFont
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    # 📁 Folder Path Settings
    # Original images path
    dir_original = "/content/drive/MyDrive/sticker_project"

    # Base path for generated results
    base_path = "/content/drive/MyDrive/Stickers_Results"

    # Subfolder paths for each style
    dir_baseline = f"{base_path}/baseline"
    dir_cartoon = f"{base_path}/cartoon"
    dir_watercolor = f"{base_path}/watercolor"
    dir_flat = f"{base_path}/flat"

    # Collage output path
    output_folder = f"{base_path}/collages"

    # 🎨 Collage Layout Settings
    groups_per_page = 5      # Number of groups (rows) per large image, recommended 5 or 6
    img_size = 512           # Uniform size for each small image (512x512)
    padding = 20             # Spacing between images
    header_height = 60       # Height of the top text header

    # Label names (displayed at the top of the collage)
    labels = ["Original", "Baseline", "Cartoon", "Watercolor", "Flat"]

def get_image_path(folder, style, number, ext_list=['.png', '.jpg', '.jpeg']):
    """Find images by trying different extensions based on the style."""
    for ext in ext_list:
        if style == "original":
            # Original image format: animal_1.png
            filename = f"animal_{number}{ext}"
        elif style == "baseline":
            # Baseline format: animal_1_baseline.png
            filename = f"animal_{number}_baseline{ext}"
        else:
            # Other styles format (cartoon, watercolor, flat): cartoon_animal_1.png
            filename = f"{style}_animal_{number}{ext}"

        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return None

def create_placeholder(size, text="Missing"):
    """Create a blank placeholder image with text if the image is missing."""
    img = Image.new('RGB', (size, size), color=(220, 220, 220))
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default()
    except:
        font = None

    # Draw text in the center
    text_bbox = draw.textbbox((0, 0), text, font=font)
    text_w = text_bbox[2] - text_bbox[0]
    text_h = text_bbox[3] - text_bbox[1]
    draw.text(((size - text_w) / 2, (size - text_h) / 2), text, fill=(255, 0, 0), font=font)
    return img

def main():
    config = Config()
    os.makedirs(config.output_folder, exist_ok=True)

    # Dataset ranges from 1 to 37
    total_images = 37
    numbers = [str(i) for i in range(1, total_images + 1)]

    # Calculate how many large collage images need to be generated
    total_pages = math.ceil(len(numbers) / config.groups_per_page)

    print(f"🚀 Starting collage generation, {len(numbers)} groups in total, will generate {total_pages} large images...")

    for page in range(total_pages):
        # Get the numbers to process for the current page
        start_idx = page * config.groups_per_page
        end_idx = min(start_idx + config.groups_per_page, len(numbers))
        current_numbers = numbers[start_idx:end_idx]

        rows = len(current_numbers)
        cols = 5 # Original + Baseline + 3 styles (Cartoon, Watercolor, Flat)

        # Calculate the dimensions of the large canvas
        canvas_width = cols * config.img_size + (cols + 1) * config.padding
        canvas_height = config.header_height + rows * config.img_size + (rows + 1) * config.padding

        # Create a white background canvas
        canvas = Image.new('RGB', (canvas_width, canvas_height), color=(255, 255, 255))
        draw = ImageDraw.Draw(canvas)
        font = ImageFont.load_default()

        # 1. Draw top labels (Header)
        for col in range(cols):
            x = config.padding + col * (config.img_size + config.padding)
            y = 20
            label = config.labels[col]

            # Simple centering calculation
            text_bbox = draw.textbbox((0, 0), label, font=font)
            text_w = text_bbox[2] - text_bbox[0]
            text_x = x + (config.img_size - text_w) / 2

            draw.text((text_x, y), label, fill=(0, 0, 0), font=font)

        # 2. Read and paste images row by row
        for row, num in enumerate(current_numbers):
            y = config.header_height + config.padding + row * (config.img_size + config.padding)

            # Define the retrieval logic for each column: (folder path, style identifier)
            img_info = [
                (config.dir_original, "original"),     # Corresponds to animal_1.png
                (config.dir_baseline, "baseline"),     # Corresponds to animal_1_baseline.png
                (config.dir_cartoon, "cartoon"),       # Corresponds to cartoon_animal_1.png
                (config.dir_watercolor, "watercolor"), # Corresponds to watercolor_animal_1.png
                (config.dir_flat, "flat")              # Corresponds to flat_animal_1.png
            ]

            for col, (folder, style) in enumerate(img_info):
                x = config.padding + col * (config.img_size + config.padding)

                # Get image path
                img_path = get_image_path(folder, style, num)

                if img_path:
                    try:
                        img = Image.open(img_path).convert('RGB')
                        img = img.resize((config.img_size, config.img_size), Image.Resampling.LANCZOS)
                    except Exception as e:
                        print(f"⚠️ Cannot read image {img_path}: {e}")
                        img = create_placeholder(config.img_size, f"Error: {num}")
                else:
                    img = create_placeholder(config.img_size, f"Missing: {num}")

                canvas.paste(img, (x, y))

        # Save the current large image
        output_filename = os.path.join(config.output_folder, f"collage_page_{page + 1}.jpg")
        canvas.save(output_filename, quality=90)
        print(f"✅ Successfully saved collage: collage_page_{page + 1}.jpg (Contains numbers {current_numbers[0]} to {current_numbers[-1]})")

    print(f"\n🎉 All collages generated! Please check {config.output_folder}.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Starting collage generation, 37 groups in total, will generate 8 large images...
✅ Successfully saved collage: collage_page_1.jpg (Contains numbers 1 to 5)
✅ Successfully saved collage: collage_page_2.jpg (Contains numbers 6 to 10)
✅ Successfully saved collage: collage_page_3.jpg (Contains numbers 11 to 15)
✅ Successfully saved collage: collage_page_4.jpg (Contains numbers 16 to 20)
✅ Successfully saved collage: collage_page_5.jpg (Contains numbers 21 to 25)
✅ Successfully saved collage: collage_page_6.jpg (Contains numbers 26 to 30)
✅ Successfully saved collage: collage_page_7.jpg (Contains numbers 31 to 35)
✅ Successfully saved collage: collage_page_8.jpg (Contains numbers 36 to 37)

🎉 All collages generated! Please check /content/drive/MyDrive/Stickers_Results/collages.


In [ ]:
# ==========================================
# SCRIPT 2: FLAT LORA ONLY (WITH WHITE BACKGROUND)
# ==========================================
import os
import json
import re
import torch
from diffusers import StableDiffusionPipeline
from google.colab import drive

# Mount Drive
try:
    drive.mount('/content/drive')
except ValueError:
    print("✅ Drive already mounted")

class Config:
    lora_path = "/content/drive/MyDrive/lora_flat"

    output_folder = "/content/drive/MyDrive/Stickers_Results/flat_white_bg"

    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    pretrained_model_name = "runwayml/stable-diffusion-v1-5"
    num_inference_steps = 50
    guidance_scale = 7.5

def load_lora_with_peft(pipe, lora_path):
    try:
        from peft import PeftModel
        print(f"\n🔄 Loading LoRA from: {lora_path}")
        pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)
        print(f"✅ LoRA loaded successfully")
        return True
    except Exception as e:
        print(f"❌ Failed to load LoRA: {e}")
        return False

def extract_number_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return match.group(1)
    return None

def load_prompts_to_dict(json_path):
    if not os.path.exists(json_path):
        print(f"❌ JSON not found: {json_path}")
        return {}

    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    if isinstance(prompts_data, dict):
        prompts_list = list(prompts_data.values())
    else:
        prompts_list = prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']

    return prompt_dict

def main():
    config = Config()

    print(f"📥 Loading base model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.pretrained_model_name,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to("cuda")
    pipe.enable_attention_slicing()
    print(f"✅ Base model loaded")

    os.makedirs(config.output_folder, exist_ok=True)

    print("📥 Loading main prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    if not prompts_main:
        print("❌ Main prompt list is empty. Exiting.")
        return

    numbers = sorted(list(prompts_main.keys()))
    total = len(numbers)
    success_lora = 0

    print(f"\n{'='*60}")
    print(f"🚀 Generating Flat LoRA Images (With White Background)")
    print(f"{'='*60}")

    # Load LoRA immediately
    if not load_lora_with_peft(pipe, config.lora_path):
        print("❌ Cannot proceed without LoRA. Exiting.")
        return

    print("\n")
    for i, number in enumerate(numbers, start=1):
        original_prompt = prompts_main[number]
        prompt_main = f"{original_prompt}, with white background"

        output_file_main = f"flat_animal_{number}.png"
        save_path_main = os.path.join(config.output_folder, output_file_main)

        print(f"[{i}/{total}] LoRA for animal_{number}.jpg")
        print(f"  📝 Prompt: {prompt_main}")

        try:
            image_main = pipe(
                prompt_main,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
            ).images[0]
            image_main.save(save_path_main)
            print(f"  ✅ Saved -> {output_file_main}\n")
            success_lora += 1
        except Exception as e:
            print(f"  ❌ Failed: {e}\n")

    print(f"{'='*60}")
    print(f"🎉 LoRA Tasks Complete! Success: {success_lora}/{total}")
    print(f"📁 Output Folder: {config.output_folder}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

✅ Base model loaded
📥 Loading main prompts...

🚀 Generating Flat LoRA Images (With White Background)

🔄 Loading LoRA from: /content/drive/MyDrive/lora_flat
✅ LoRA loaded successfully


[1/37] LoRA for animal_1.jpg
  📝 Prompt: A sleek black sks cat sitting. Sticker style., with white background


...

  ✅ Saved -> flat_animal_7.png

[36/37] LoRA for animal_8.jpg
  📝 Prompt: A curious sks dog sitting and looking at the camera, sticker style., with white background


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> flat_animal_8.png

[37/37] LoRA for animal_9.jpg
  📝 Prompt: A fluffy black and white Border Collie sks dog sitting and looking at camera, sticker style., with white background


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Saved -> flat_animal_9.png

🎉 LoRA Tasks Complete! Success: 37/37
📁 Output Folder: /content/drive/MyDrive/Stickers_Results/flat_white_bg


In [5]:
# ==========================================
# SCRIPT: LoRA Steps Comparison Grid (PEFT)
# ==========================================
import os
import json
import re
import torch
from PIL import Image, ImageDraw, ImageFont
from diffusers import StableDiffusionPipeline
from peft import PeftModel

# If running in Colab, uncomment the following two lines to mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("⚠️ Not in Colab environment, skipping Google Drive mount.")

class Config:
    # Base model path
    pretrained_model_name = "runwayml/stable-diffusion-v1-5"

    # Path to your JSON prompt file
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    # Output folder for grid and individual images
    output_folder = "/content/drive/MyDrive/Stickers_Results/grid_output"

    # Base path for LoRA training steps folders
    lora_base_path = "/content/drive/MyDrive/lora_flat_step"

    # List of steps you want to test (corresponds to i in lora_steps_{i})
    steps = [200, 400, 600, 800, 1000, 1200]

    # Generation parameters
    num_inference_steps = 50
    guidance_scale = 7.5
    seed = 42 # Fixed seed to ensure consistent composition for easy LoRA comparison

    # Animal numbers to test (to prevent the grid from being too large, you can select just the first few, e.g., [1, 2, 3, 4])
    # If set to None, test all animals in the JSON
    test_animal_numbers = [1, 2, 3, 4]

def extract_number_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return int(match.group(1))
    return None

def load_prompts_to_dict(json_path):
    if not os.path.exists(json_path):
        print(f"❌ Cannot find JSON file: {json_path}")
        return {}
    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    prompts_list = list(prompts_data.values()) if isinstance(prompts_data, dict) else prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']
    return prompt_dict

def create_grid(images_dict, steps, animal_numbers, output_path):
    """
    Stitch generated images into a grid
    images_dict: Format is {(animal_number, step): PIL.Image}
    """
    if not images_dict:
        print("❌ No images available to create the grid.")
        return

    # Get the dimensions of a single image
    sample_img = list(images_dict.values())[0]
    w, h = sample_img.size

    # Set the number of rows and columns for the grid (including margin for headers)
    header_height = 50
    sidebar_width = 100

    grid_width = sidebar_width + len(steps) * w
    grid_height = header_height + len(animal_numbers) * h

    grid_img = Image.new('RGB', (grid_width, grid_height), color='white')
    draw = ImageDraw.Draw(grid_img)

    # Try to load a font, use default if it fails
    try:
        font = ImageFont.truetype("arial.ttf", 24)
    except:
        font = ImageFont.load_default()

    # Draw column headers (Steps)
    for col, step in enumerate(steps):
        text = f"Step {step}"
        # Simple calculation for text centering
        text_x = sidebar_width + col * w + (w // 2) - 40
        text_y = 15
        draw.text((text_x, text_y), text, fill="black", font=font)

    # Draw row headers (Animal Number) and paste images
    for row, animal_num in enumerate(animal_numbers):
        # Row header
        text = f"Animal {animal_num}"
        text_x = 10
        text_y = header_height + row * h + (h // 2) - 10
        draw.text((text_x, text_y), text, fill="black", font=font)

        # Paste all step images for this row
        for col, step in enumerate(steps):
            img = images_dict.get((animal_num, step))
            if img:
                paste_x = sidebar_width + col * w
                paste_y = header_height + row * h
                grid_img.paste(img, (paste_x, paste_y))

    grid_img.save(output_path)
    print(f"\n🎉 Grid image successfully saved to: {output_path}")

def main():
    config = Config()
    os.makedirs(config.output_folder, exist_ok=True)

    print("📥 Loading prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    # Filter animal numbers to test
    if config.test_animal_numbers:
        animal_numbers = [n for n in config.test_animal_numbers if n in prompts_main]
    else:
        animal_numbers = sorted(list(prompts_main.keys()))

    if not animal_numbers:
        print("❌ No matching prompts found, please check the JSON file.")
        return

    # Dictionary to store all generated images, format: {(animal_number, step): PIL.Image}
    generated_images = {}

    print(f"\n{'='*60}")
    print(f"🚀 Starting to generate LoRA steps comparison grid (Fixed seed: {config.seed})")
    print(f"{'='*60}")

    # Outer loop: Iterate through each Step
    for step in config.steps:
        # ⚠️ Folder concatenation logic updated here: lora_steps_{step}
        lora_path = os.path.join(config.lora_base_path, f"lora_step_{step}")

        if not os.path.exists(lora_path):
            print(f"⚠️ Path not found: {lora_path}, skipping Step {step}")
            continue

        print(f"\n🔄 [Step {step}] Loading clean base model...")
        # Reload clean base model when switching steps to prevent residual LoRA weights (VRAM cleanup)
        if 'pipe' in locals():
            del pipe
            torch.cuda.empty_cache()

        pipe = StableDiffusionPipeline.from_pretrained(
            config.pretrained_model_name,
            torch_dtype=torch.float16,
            safety_checker=None,
        ).to("cuda")
        pipe.enable_attention_slicing()

        print(f"🔄 [Step {step}] Loading LoRA weights using PEFT...")
        try:
            pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)
            print(f"✅ [Step {step}] LoRA loaded successfully!")
        except Exception as e:
            print(f"❌ [Step {step}] Failed to load LoRA: {e}, skipping this step.")
            continue

        # Inner loop: Iterate through each animal prompt to generate images
        for animal_num in animal_numbers:
            prompt = prompts_main[animal_num]
            print(f"  🎨 Generating Animal {animal_num} (Step {step})...")

            # Set fixed random seed
            generator = torch.Generator(device="cuda").manual_seed(config.seed)

            try:
                image = pipe(
                    prompt,
                    num_inference_steps=config.num_inference_steps,
                    guidance_scale=config.guidance_scale,
                    generator=generator
                ).images[0]

                # Save single image (optional, for easy individual viewing)
                single_img_path = os.path.join(config.output_folder, f"animal_{animal_num}_step_{step}.png")
                image.save(single_img_path)

                # Store in dictionary for later grid assembly
                generated_images[(animal_num, step)] = image

            except Exception as e:
                print(f"  ❌ Generation failed: {e}")

    # All images generated, starting to assemble the grid
    print(f"\n{'='*60}")
    print(f"🖼️ Assembling comparison grid...")
    grid_output_path = os.path.join(config.output_folder, "lora_steps_comparison_grid.png")
    create_grid(generated_images, config.steps, animal_numbers, grid_output_path)
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading prompts...

🚀 Starting to generate LoRA steps comparison grid (Fixed seed: 42)

🔄 [Step 200] Loading clean base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

🔄 [Step 200] Loading LoRA weights using PEFT...
✅ [Step 200] LoRA loaded successfully!
  🎨 Generating Animal 1 (Step 200)...


...

  🎨 Generating Animal 3 (Step 1200)...


  0%|          | 0/50 [00:00<?, ?it/s]

  🎨 Generating Animal 4 (Step 1200)...


  0%|          | 0/50 [00:00<?, ?it/s]


🖼️ Assembling comparison grid...

🎉 Grid image successfully saved to: /content/drive/MyDrive/Stickers_Results/grid_output/lora_steps_comparison_grid.png


In [6]:
# ==========================================
# SCRIPT: LoRA Steps Comparison Grid - Cartoon Style
# ==========================================
import os
import json
import re
import torch
from PIL import Image, ImageDraw, ImageFont
from diffusers import StableDiffusionPipeline

# If running in Colab, uncomment the following two lines to mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("⚠️ Not in Colab environment, skipping Google Drive mount.")

class Config:
    # Base model path
    pretrained_model_name = "runwayml/stable-diffusion-v1-5"

    # Path to your JSON prompt file
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    # Output folder for grid and individual images
    output_folder = "/content/drive/MyDrive/Stickers_Results/cartoon_grid_output"

    # Base path for LoRA training steps folders
    lora_base_path = "/content/drive/MyDrive/lora_cartoon_step"

    # List of steps you want to test (corresponds to subfolder names 200, 400...)
    steps = [200, 400, 600, 800, 1000, 1200]

    # Generation parameters
    num_inference_steps = 50
    guidance_scale = 7.5
    seed = 42 # Fixed seed to ensure consistent composition for easy LoRA comparison

    # Animal numbers to test (to prevent the grid from being too large, you can select just the first few, e.g., [1, 2, 3, 4])
    # If set to None, test all animals in the JSON
    test_animal_numbers = [1, 2, 3, 4]

def extract_number_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return int(match.group(1))
    return None

def load_prompts_to_dict(json_path):
    if not os.path.exists(json_path):
        print(f"❌ Cannot find JSON file: {json_path}")
        return {}
    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    prompts_list = list(prompts_data.values()) if isinstance(prompts_data, dict) else prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']
    return prompt_dict

def create_grid(images_dict, steps, animal_numbers, output_path):
    """
    Stitch generated images into a grid
    images_dict: Format is {(animal_number, step): PIL.Image}
    """
    if not images_dict:
        print("❌ No images available to create the grid.")
        return

    # Get the dimensions of a single image
    sample_img = list(images_dict.values())[0]
    w, h = sample_img.size

    # Set the number of rows and columns for the grid (including margin for headers)
    header_height = 50
    sidebar_width = 100

    grid_width = sidebar_width + len(steps) * w
    grid_height = header_height + len(animal_numbers) * h

    grid_img = Image.new('RGB', (grid_width, grid_height), color='white')
    draw = ImageDraw.Draw(grid_img)

    # Try to load a font, use default if it fails
    try:
        font = ImageFont.truetype("arial.ttf", 24)
    except:
        font = ImageFont.load_default()

    # Draw column headers (Steps)
    for col, step in enumerate(steps):
        text = f"Step {step}"
        # Simple calculation for text centering
        text_x = sidebar_width + col * w + (w // 2) - 40
        text_y = 15
        draw.text((text_x, text_y), text, fill="black", font=font)

    # Draw row headers (Animal Number) and paste images
    for row, animal_num in enumerate(animal_numbers):
        # Row header
        text = f"Animal {animal_num}"
        text_x = 10
        text_y = header_height + row * h + (h // 2) - 10
        draw.text((text_x, text_y), text, fill="black", font=font)

        # Paste all step images for this row
        for col, step in enumerate(steps):
            img = images_dict.get((animal_num, step))
            if img:
                paste_x = sidebar_width + col * w
                paste_y = header_height + row * h
                grid_img.paste(img, (paste_x, paste_y))

    grid_img.save(output_path)
    print(f"\n🎉 Grid image successfully saved to: {output_path}")

def main():
    config = Config()
    os.makedirs(config.output_folder, exist_ok=True)

    print("📥 Loading prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    # Filter animal numbers to test
    if config.test_animal_numbers:
        animal_numbers = [n for n in config.test_animal_numbers if n in prompts_main]
    else:
        animal_numbers = sorted(list(prompts_main.keys()))

    if not animal_numbers:
        print("❌ No matching prompts found, please check the JSON file.")
        return

    # Dictionary to store all generated images, format: {(animal_number, step): PIL.Image}
    generated_images = {}

    print(f"\n{'='*60}")
    print(f"🚀 Starting to generate Cartoon LoRA steps comparison grid (Fixed seed: {config.seed})")
    print(f"{'='*60}")

    # Outer loop: Iterate through each Step
    for step in config.steps:
        lora_path = os.path.join(config.lora_base_path, str(step))

        if not os.path.exists(lora_path):
            print(f"⚠️ Path not found: {lora_path}, skipping Step {step}")
            continue

        print(f"\n🔄 [Step {step}] Loading clean base model...")
        # Reload clean base model when switching steps to prevent residual LoRA weights (VRAM cleanup)
        if 'pipe' in locals():
            del pipe
            torch.cuda.empty_cache()

        pipe = StableDiffusionPipeline.from_pretrained(
            config.pretrained_model_name,
            torch_dtype=torch.float16,
            safety_checker=None,
        ).to("cuda")
        pipe.enable_attention_slicing()

        print(f"🔄 [Step {step}] Loading LoRA weights using Diffusers native method...")
        try:
            # 👇 Modification: Removed PEFT, directly using diffusers' built-in load_lora_weights
            pipe.load_lora_weights(lora_path)
            print(f"✅ [Step {step}] LoRA loaded successfully!")
        except Exception as e:
            print(f"❌ [Step {step}] Failed to load LoRA: {e}, skipping this step.")
            continue

        # Inner loop: Iterate through each animal prompt to generate images
        for animal_num in animal_numbers:
            prompt = prompts_main[animal_num]
            print(f"  🎨 Generating Animal {animal_num} (Step {step})...")

            # Set fixed random seed
            generator = torch.Generator(device="cuda").manual_seed(config.seed)

            try:
                image = pipe(
                    prompt,
                    num_inference_steps=config.num_inference_steps,
                    guidance_scale=config.guidance_scale,
                    generator=generator
                ).images[0]

                # Save single image (optional, for easy individual viewing)
                single_img_path = os.path.join(config.output_folder, f"cartoon_animal_{animal_num}_step_{step}.png")
                image.save(single_img_path)

                # Store in dictionary for later grid assembly
                generated_images[(animal_num, step)] = image

            except Exception as e:
                print(f"  ❌ Generation failed: {e}")

    # All images generated, starting to assemble the grid
    print(f"\n{'='*60}")
    print(f"🖼️ Assembling comparison grid...")
    grid_output_path = os.path.join(config.output_folder, "lora_cartoon_steps_comparison_grid.png")
    create_grid(generated_images, config.steps, animal_numbers, grid_output_path)
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading prompts...

🚀 Starting to generate Cartoon LoRA steps comparison grid (Fixed seed: 42)

🔄 [Step 200] Loading clean base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

🔄 [Step 200] Loading LoRA weights using Diffusers native method...


...

  🎨 Generating Animal 3 (Step 1200)...


  0%|          | 0/50 [00:00<?, ?it/s]

  🎨 Generating Animal 4 (Step 1200)...


  0%|          | 0/50 [00:00<?, ?it/s]


🖼️ Assembling comparison grid...

🎉 Grid image successfully saved to: /content/drive/MyDrive/Stickers_Results/cartoon_grid_output/lora_cartoon_steps_comparison_grid.png


In [7]:
# ==========================================
# SCRIPT: LoRA Steps Comparison Grid - Watercolor Style
# ==========================================
import os
import json
import re
import torch
from PIL import Image, ImageDraw, ImageFont
from diffusers import StableDiffusionPipeline

# If running in Colab, uncomment the following two lines to mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("⚠️ Not in Colab environment, skipping Google Drive mount.")

class Config:
    # Base model path
    pretrained_model_name = "runwayml/stable-diffusion-v1-5"

    # Path to your JSON prompt file
    json_path = "/content/drive/MyDrive/batch_sticker_prompts.json"

    # Output folder for grid and individual images (modified to watercolor)
    output_folder = "/content/drive/MyDrive/Stickers_Results/watercolor_grid_output"

    # Base path for LoRA training steps folders (modified to watercolor)
    lora_base_path = "/content/drive/MyDrive/lora_watercolor_step"

    # List of steps you want to test
    steps = [200, 400, 600, 800, 1000, 1200]

    # Generation parameters
    num_inference_steps = 50
    guidance_scale = 7.5
    seed = 42 # Fixed seed to ensure consistent composition for easy LoRA comparison

    # Animal numbers to test (to prevent the grid from being too large, you can select just the first few, e.g., [1, 2, 3, 4])
    # If set to None, test all animals in the JSON
    test_animal_numbers = [1, 2, 3, 4]

def extract_number_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = re.search(r'animal_(\d+)', base_name)
    if match:
        return int(match.group(1))
    return None

def load_prompts_to_dict(json_path):
    if not os.path.exists(json_path):
        print(f"❌ Cannot find JSON file: {json_path}")
        return {}
    with open(json_path, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    prompts_list = list(prompts_data.values()) if isinstance(prompts_data, dict) else prompts_data

    prompt_dict = {}
    for item in prompts_list:
        number = extract_number_from_filename(item['file_name'])
        if number is not None:
            prompt_dict[number] = item['sd_sticker_prompt']
    return prompt_dict

def create_grid(images_dict, steps, animal_numbers, output_path):
    """
    Stitch generated images into a grid
    images_dict: Format is {(animal_number, step): PIL.Image}
    """
    if not images_dict:
        print("❌ No images available to create the grid.")
        return

    # Get the dimensions of a single image
    sample_img = list(images_dict.values())[0]
    w, h = sample_img.size

    # Set the number of rows and columns for the grid (including margin for headers)
    header_height = 50
    sidebar_width = 100

    grid_width = sidebar_width + len(steps) * w
    grid_height = header_height + len(animal_numbers) * h

    grid_img = Image.new('RGB', (grid_width, grid_height), color='white')
    draw = ImageDraw.Draw(grid_img)

    # Try to load a font, use default if it fails
    try:
        font = ImageFont.truetype("arial.ttf", 24)
    except:
        font = ImageFont.load_default()

    # Draw column headers (Steps)
    for col, step in enumerate(steps):
        text = f"Step {step}"
        # Simple calculation for text centering
        text_x = sidebar_width + col * w + (w // 2) - 40
        text_y = 15
        draw.text((text_x, text_y), text, fill="black", font=font)

    # Draw row headers (Animal Number) and paste images
    for row, animal_num in enumerate(animal_numbers):
        # Row header
        text = f"Animal {animal_num}"
        text_x = 10
        text_y = header_height + row * h + (h // 2) - 10
        draw.text((text_x, text_y), text, fill="black", font=font)

        # Paste all step images for this row
        for col, step in enumerate(steps):
            img = images_dict.get((animal_num, step))
            if img:
                paste_x = sidebar_width + col * w
                paste_y = header_height + row * h
                grid_img.paste(img, (paste_x, paste_y))

    grid_img.save(output_path)
    print(f"\n🎉 Grid image successfully saved to: {output_path}")

def main():
    config = Config()
    os.makedirs(config.output_folder, exist_ok=True)

    print("📥 Loading prompts...")
    prompts_main = load_prompts_to_dict(config.json_path)

    # Filter animal numbers to test
    if config.test_animal_numbers:
        animal_numbers = [n for n in config.test_animal_numbers if n in prompts_main]
    else:
        animal_numbers = sorted(list(prompts_main.keys()))

    if not animal_numbers:
        print("❌ No matching prompts found, please check the JSON file.")
        return

    # Dictionary to store all generated images, format: {(animal_number, step): PIL.Image}
    generated_images = {}

    print(f"\n{'='*60}")
    print(f"🚀 Starting to generate Watercolor LoRA steps comparison grid (Fixed seed: {config.seed})")
    print(f"{'='*60}")

    # Outer loop: Iterate through each Step
    for step in config.steps:
        # 👇 Modification: Adapt to the checkpoint-{step} folder naming format
        folder_name = f"checkpoint-{step}"
        lora_path = os.path.join(config.lora_base_path, folder_name)

        if not os.path.exists(lora_path):
            print(f"⚠️ Path not found: {lora_path}, skipping Step {step}")
            continue

        print(f"\n🔄 [Step {step}] Loading clean base model...")
        # Reload clean base model when switching steps to prevent residual LoRA weights (VRAM cleanup)
        if 'pipe' in locals():
            del pipe
            torch.cuda.empty_cache()

        pipe = StableDiffusionPipeline.from_pretrained(
            config.pretrained_model_name,
            torch_dtype=torch.float16,
            safety_checker=None,
        ).to("cuda")
        pipe.enable_attention_slicing()

        print(f"🔄 [Step {step}] Loading LoRA weights using Diffusers native method...")
        try:
            # Use diffusers' built-in load_lora_weights
            pipe.load_lora_weights(lora_path)
            print(f"✅ [Step {step}] LoRA loaded successfully!")
        except Exception as e:
            print(f"❌ [Step {step}] Failed to load LoRA: {e}, skipping this step.")
            continue

        # Inner loop: Iterate through each animal prompt to generate images
        for animal_num in animal_numbers:
            prompt = prompts_main[animal_num]
            print(f"  🎨 Generating Animal {animal_num} (Step {step})...")

            # Set fixed random seed
            generator = torch.Generator(device="cuda").manual_seed(config.seed)

            try:
                image = pipe(
                    prompt,
                    num_inference_steps=config.num_inference_steps,
                    guidance_scale=config.guidance_scale,
                    generator=generator
                ).images[0]

                # Save single image (modify filename to watercolor)
                single_img_path = os.path.join(config.output_folder, f"watercolor_animal_{animal_num}_step_{step}.png")
                image.save(single_img_path)

                # Store in dictionary for later grid assembly
                generated_images[(animal_num, step)] = image

            except Exception as e:
                print(f"  ❌ Generation failed: {e}")

    # All images generated, starting to assemble the grid
    print(f"\n{'='*60}")
    print(f"🖼️ Assembling comparison grid...")
    grid_output_path = os.path.join(config.output_folder, "lora_watercolor_steps_comparison_grid.png")
    create_grid(generated_images, config.steps, animal_numbers, grid_output_path)
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Loading prompts...

🚀 Starting to generate Watercolor LoRA steps comparison grid (Fixed seed: 42)

🔄 [Step 200] Loading clean base model...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

🔄 [Step 200] Loading LoRA weights using Diffusers native method...


...

  🎨 Generating Animal 3 (Step 1200)...


  0%|          | 0/50 [00:00<?, ?it/s]

  🎨 Generating Animal 4 (Step 1200)...


  0%|          | 0/50 [00:00<?, ?it/s]


🖼️ Assembling comparison grid...

🎉 Grid image successfully saved to: /content/drive/MyDrive/Stickers_Results/watercolor_grid_output/lora_watercolor_steps_comparison_grid.png
